In [2]:
import pandas as pd

df = pd.read_csv("data/cs_inquiries.csv", encoding="utf-8-sig")
print(df[["content", "category_hint"]].head(3).to_string(index=False))

                          content category_hint
 카드 결제가 두 번 청구된 것 같아요. 확인 부탁드립니다.            결제
        단순 변심인데 반품 배송비는 누가 부담하나요?            환불
선크림 SPF50 유통기한이 얼마나 남았는지 알 수 있나요?          상품문의


In [3]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# NVIDIA(OpenAI 호환 API) 클라이언트 초기화
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

# 역할 + 지시 + 맥락(제약)을 시스템 메시지에 담는다
ROLE = (
    "너는 승승장구몰의 친절한 CS 상담원이다. "          # 역할(Role)
    "고객 문의에 존댓말로 공감하며 간결하게 답하라. "    # 지시(Instruction)
    "확실하지 않은 정보는 '확인 후 안내드리겠습니다'라고 답하라."  # 맥락(Context: 제약)
)

def reply(content: str) -> str:
    """고객 문의에 ROLE 페르소나로 정중한 답변을 생성한다."""
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": ROLE},
            {"role": "user", "content": f"고객 문의: {content}"}
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content

In [5]:
print(reply("카드 결제가 두 번 청구된 것 같아요. 확인 부탁드립니다."))

네, 카드 결제가 두 번 청구된 것 같아 불편을 드려 죄송합니다.  
현재 계정 정보를 확인 후 정확한 사유와 해결 방안을 안내드리겠습니다.  
잠시만 기다려 주시고, 확인 후 바로 연락드리겠습니다.


In [6]:
CATEGORIES = ["배송", "환불", "교환", "결제", "상품문의", "칭찬", "불만"]

# few-shot = 정답 예시 몇 개를 먼저 보여주고 같은 식으로 답하게 하는 기법.
FEWSHOT = """다음 고객 문의를 아래 7개 중 정확히 하나로 분류하라.
카테고리: 배송 / 환불 / 교환 / 결제 / 상품문의 / 칭찬 / 불만
카테고리 이름 한 단어만 출력하라(다른 말 금지).

[예시]
문의: 반품하면 배송비는 누가 부담하나요?           → 환불
문의: 색상이 사진과 달라요. 다른 색으로 바꿔주세요.  → 교환
문의: 상담원분이 정말 친절하셨어요. 감사합니다.      → 칭찬
문의: 카드가 두 번 청구됐어요.                      → 결제
"""

In [15]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

CATEGORIES = ["배송", "환불", "교환", "결제", "상품문의", "칭찬", "불만"]

def classify(content: str) -> str:
    """문의 한 건을 7개 카테고리 중 하나로 분류한다(few-shot 사용)."""
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{FEWSHOT}\n[분류할 문의]\n문의: {content} →"}
        ],
    )
    out = resp.choices[0].message.content.strip()
    for c in CATEGORIES:        # 군더더기가 붙어도 7개 중 포함된 단어를 골라낸다
        if c in out:
            return c
    return "기타"

In [16]:
print(classify("카드가 두 번 청구됐어요"))        # → 결제
print(classify("포장이 찢어진 채로 왔어요"))       # → 불만 또는 교환
print(classify("이 제품 방수 되나요?"))            # → 상품문의

결제
불만
상품문의


In [ ]:
import json
from openai import OpenAI
from dotenv import load_dotenv

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

prompt = (
    "다음 고객 문의를 분석해 JSON으로만 답하라.\n"
    'key: category, urgent, summary\n'
    "문의: 어제 받은 제품이 박살나서 왔어요."
)
resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": prompt}]
)
data = json.loads(resp.choices[0].message.content)   # 운이 나쁘면 모델이 설명을 덧붙여 실패할 수 있음

In [18]:
import json
from openai import OpenAI
from dotenv import load_dotenv

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

def triage(content: str) -> dict:
    """문의를 분석해 category/urgent/summary 를 담은 dict로 돌려준다."""
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "너는 고객 문의를 분석하여 JSON 형식으로만 답변하는 도우미다.\n"
                    "반드시 다음 키를 포함한 JSON 객체만 응답하라:\n"
                    "- category: (배송/환불/교환/결제/상품문의/칭찬/불만 중 하나)\n"
                    "- urgent: (true/false)\n"
                    "- summary: (20자 이내 한국어)"
                )
            },
            {
                "role": "user",
                "content": f"문의: {content}"
            }
        ],
        temperature=0,
        response_format={"type": "json_object"},   # ← JSON만 출력하도록 강제
    )
    return json.loads(resp.choices[0].message.content)   # JSON 문자열 → 파이썬 dict

r = triage("어제 받은 제품이 박살나서 왔어요. 당장 환불해주세요!")
print(r)
print("긴급?", r["urgent"], "/ 분류:", r["category"])

{'category': '환불', 'urgent': True, 'summary': '제품 파손, 환불 요청'}
긴급? True / 분류: 환불


In [19]:
import os
import pathlib
import json
import pandas as pd
from openai import OpenAI

# OpenAI 클라이언트 초기화
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

# 경로 설정 및 CSV 로드 (cs_inquiries.csv = 고객 문의 60건. category_hint 컬럼이 정답 라벨)
DATA_PATH = pathlib.Path("./data")
df = pd.read_csv(DATA_PATH / "cs_inquiries.csv")

print("문의 건수:", len(df))
print(df[["content", "category_hint"]].head(3).to_string(index=False))

문의 건수: 60
                          content category_hint
 카드 결제가 두 번 청구된 것 같아요. 확인 부탁드립니다.            결제
        단순 변심인데 반품 배송비는 누가 부담하나요?            환불
선크림 SPF50 유통기한이 얼마나 남았는지 알 수 있나요?          상품문의


In [20]:
ROLE = (
    "너는 승승장구몰의 친절한 CS 상담원이다. "
    "고객 문의에 존댓말로 공감하며 간결하게 답하라. "
    "확실하지 않은 정보는 '확인 후 안내드리겠습니다'라고 답하라."
)

def reply(content: str) -> str:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": ROLE},
            {"role": "user", "content": f"고객 문의: {content}"}
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content

sample = df.iloc[0]["content"]
print("문의:", sample)
print("답변:", reply(sample))

문의: 카드 결제가 두 번 청구된 것 같아요. 확인 부탁드립니다.
답변: 네, 확인해 보겠습니다.  
주문 번호와 결제 일시(혹은 카드 명세서에 표시된 금액)를 알려주시면 보다 정확히 확인해 드릴 수 있어요.  
잠시만 기다려 주시고, 확인 후 바로 안내드리겠습니다. 감사합니다.


In [21]:
CATEGORIES = ["배송", "환불", "교환", "결제", "상품문의", "칭찬", "불만"]

FEWSHOT = """다음 고객 문의를 아래 7개 중 정확히 하나로 분류하라.
카테고리: 배송 / 환불 / 교환 / 결제 / 상품문의 / 칭찬 / 불만
카테고리 이름 한 단어만 출력하라(다른 말 금지).

[예시]
문의: 반품하면 배송비는 누가 부담하나요?           → 환불
문의: 색상이 사진과 달라요. 다른 색으로 바꿔주세요.  → 교환
문의: 상담원분이 정말 친절하셨어요. 감사합니다.      → 칭찬
문의: 카드가 두 번 청구됐어요.                      → 결제
"""

def classify(content: str) -> str:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{FEWSHOT}\n[분류할 문의]\n문의: {content} →"}
        ],
        temperature=0,  # 일관성을 위해 0 설정
    )
    out = resp.choices[0].message.content.strip()
    for c in CATEGORIES:
        if c in out:
            return c
    return "기타"

# 60건 전부 분류 → 정답 라벨(category_hint)과 비교
df["pred"] = df["content"].apply(classify)
correct = (df["pred"] == df["category_hint"]).sum()
print(f"분류 정확도: {correct/len(df):.1%}  ({correct}/{len(df)})")

# 틀린 사례 몇 개 출력
wrong = df[df["pred"] != df["category_hint"]]
if len(wrong) > 0:
    print("\n틀린 사례(일부):")
    print(wrong[["content", "category_hint", "pred"]].head().to_string(index=False))

분류 정확도: 100.0%  (60/60)


In [22]:
CATEGORIES = ["배송", "환불", "교환", "결제", "상품문의", "칭찬", "불만"]

FEWSHOT = """다음 고객 문의를 아래 7개 중 정확히 하나로 분류하라.
카테고리: 배송 / 환불 / 교환 / 결제 / 상품문의 / 칭찬 / 불만
카테고리 이름 한 단어만 출력하라(다른 말 금지)."""

def classify(content: str) -> str:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{FEWSHOT}\n[분류할 문의]\n문의: {content} →"}
        ],
        temperature=0,  # 일관성을 위해 0 설정
    )
    out = resp.choices[0].message.content.strip()
    for c in CATEGORIES:
        if c in out:
            return c
    return "기타"

# 60건 전부 분류 → 정답 라벨(category_hint)과 비교
df["pred"] = df["content"].apply(classify)
correct = (df["pred"] == df["category_hint"]).sum()
print(f"분류 정확도: {correct/len(df):.1%}  ({correct}/{len(df)})")

# 틀린 사례 몇 개 출력
wrong = df[df["pred"] != df["category_hint"]]
if len(wrong) > 0:
    print("\n틀린 사례(일부):")
    print(wrong[["content", "category_hint", "pred"]].head().to_string(index=False))


분류 정확도: 98.3%  (59/60)

틀린 사례(일부):
                     content category_hint pred
주문한 상품과 다른 상품이 배송됐어요. 황당하네요.            불만   배송


In [23]:
def triage(content: str) -> dict:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "너는 고객 문의를 분석하여 JSON 형식으로만 답변하는 도우미다.\n"
                    "반드시 다음 키를 포함한 JSON 객체만 응답하라:\n"
                    "- category: (배송/환불/교환/결제/상품문의/칭찬/불만 중 하나)\n"
                    "- urgent: (true/false)\n"
                    "- summary: (20자 이내 한국어)"
                )
            },
            {
                "role": "user",
                "content": f"문의: {content}"
            }
        ],
        temperature=0,
        response_format={"type": "json_object"},  # ← JSON 출력 강제
    )
    return json.loads(resp.choices[0].message.content)

r = triage("어제 받은 제품이 박살나서 왔어요. 당장 환불해주세요!")
print(r)
print("긴급?", r["urgent"], "/ 분류:", r["category"])

{'category': '환불', 'urgent': True, 'summary': '제품 파손, 환불 요청'}
긴급? True / 분류: 환불


In [27]:
import os
import pathlib
import pandas as pd
from collections import Counter
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"
DATA_PATH = pathlib.Path("./data")

CATEGORIES = ["배송", "환불", "교환", "결제", "상품문의", "칭찬", "불만"]

FEWSHOT = """다음 고객 문의를 아래 7개 중 정확히 하나로 분류하라.
카테고리: 배송 / 환불 / 교환 / 결제 / 상품문의 / 칭찬 / 불만
카테고리 이름 한 단어만 출력하라(다른 말 금지).
"""
# [예시]
# 문의: 반품하면 배송비는 누가 부담하나요?           → 환불
# 문의: 색상이 사진과 달라요. 다른 색으로 바꿔주세요.  → 교환
# 문의: 상담원분이 정말 친절하셨어요. 감사합니다.      → 칭찬
# 문의: 카드가 두 번 청구됐어요.                      → 결제


def classify(content: str) -> str:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{FEWSHOT}\n[분류할 문의]\n문의: {content} →"}
        ],
        temperature=0,
    )
    out = resp.choices[0].message.content.strip()
    for c in CATEGORIES:
        if c in out:
            return c
    return "기타"

df = pd.read_csv(DATA_PATH / "cs_inquiries.csv", encoding="utf-8-sig")
df["pred"] = df["content"].apply(classify)

correct = (df["pred"] == df["category_hint"]).sum()
print(f"전체 정확도: {correct/len(df):.1%}  ({correct}/{len(df)})")

# [핵심] 틀린 케이스만 모아서 '왜 틀렸나'를 사람이 읽을 수 있게 출력
wrong = df[df["pred"] != df["category_hint"]]
print(f"틀린 케이스: {len(wrong)}건")
for i, (_, r) in enumerate(wrong.iterrows(), 1):
    print(f"[{i}] 정답={r['category_hint']} / 예측={r['pred']}")
    print(f"     내용: {r['content']}")

전체 정확도: 98.3%  (59/60)
틀린 케이스: 1건
[1] 정답=불만 / 예측=배송
     내용: 주문한 상품과 다른 상품이 배송됐어요. 황당하네요.


In [28]:
import os
import pathlib
import pandas as pd
from collections import Counter
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"
DATA_PATH = pathlib.Path("./data")

CATEGORIES = ["배송", "환불", "교환", "결제", "상품문의", "칭찬", "불만"]

FEWSHOT = """다음 고객 문의를 아래 7개 중 정확히 하나로 분류하라.
카테고리: 배송 / 환불 / 교환 / 결제 / 상품문의 / 칭찬 / 불만
카테고리 이름 한 단어만 출력하라(다른 말 금지).

[예시]
문의: 반품하면 배송비는 누가 부담하나요?           → 환불
문의: 색상이 사진과 달라요. 다른 색으로 바꿔주세요.  → 교환
문의: 상담원분이 정말 친절하셨어요. 감사합니다.      → 칭찬
문의: 카드가 두 번 청구됐어요.                      → 결제
"""

def classify(content: str) -> str:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{FEWSHOT}\n[분류할 문의]\n문의: {content} →"}
        ],
        temperature=0,
    )
    out = resp.choices[0].message.content.strip()
    for c in CATEGORIES:
        if c in out:
            return c
    return "기타"

df = pd.read_csv(DATA_PATH / "cs_inquiries.csv", encoding="utf-8-sig")
df["pred"] = df["content"].apply(classify)

correct = (df["pred"] == df["category_hint"]).sum()
print(f"전체 정확도: {correct/len(df):.1%}  ({correct}/{len(df)})")

# [핵심] 틀린 케이스만 모아서 '왜 틀렸나'를 사람이 읽을 수 있게 출력
wrong = df[df["pred"] != df["category_hint"]]
print(f"틀린 케이스: {len(wrong)}건")
for i, (_, r) in enumerate(wrong.iterrows(), 1):
    print(f"[{i}] 정답={r['category_hint']} / 예측={r['pred']}")
    print(f"     내용: {r['content']}")

전체 정확도: 100.0%  (60/60)
틀린 케이스: 0건


In [31]:
import os
import pathlib
import pandas as pd
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"
DATA_PATH = pathlib.Path("./data")

df = pd.read_csv(DATA_PATH / "cs_inquiries.csv", encoding="utf-8-sig")
CATEGORIES = ["배송", "환불", "교환", "결제", "상품문의", "칭찬", "불만"]

FEWSHOT_BASE = """다음 고객 문의를 아래 7개 중 정확히 하나로 분류하라.
카테고리: 배송 / 환불 / 교환 / 결제 / 상품문의 / 칭찬 / 불만
카테고리 이름 한 단어만 출력하라(다른 말 금지).

[예시]
문의: 반품하면 배송비는 누가 부담하나요?           → 환불
문의: 색상이 사진과 달라요. 다른 색으로 바꿔주세요.  → 교환
문의: 상담원분이 정말 친절하셨어요. 감사합니다.      → 칭찬
문의: 카드가 두 번 청구됐어요.                      → 결제
"""

# 경계 예시 2개 추가
FEWSHOT_PLUS = FEWSHOT_BASE + """문의: 배송이 자꾸 늦어서 너무 불편해요.   → 불만
문의: 이 제품 방수 되나요?                  → 상품문의
"""

def make_classifier(fewshot):
    def classify(content):
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "user", "content": f"{fewshot}\n[분류할 문의]\n문의: {content} →"}
            ],
            temperature=0,
        )
        out = resp.choices[0].message.content.strip()
        return next((c for c in CATEGORIES if c in out), "기타")
    return classify

def accuracy(classify):
    pred = df["content"].apply(classify)
    return (pred == df["category_hint"]).mean()

print(f"예시 추가 전: {accuracy(make_classifier(FEWSHOT_BASE)):.1%}")
print(f"예시 추가 후: {accuracy(make_classifier(FEWSHOT_PLUS)):.1%}")

예시 추가 전: 100.0%
예시 추가 후: 93.3%


In [29]:
import os
import pathlib
import json
import pandas as pd
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"
DATA_PATH = pathlib.Path("./data")

df = pd.read_csv(DATA_PATH / "cs_inquiries.csv", encoding="utf-8-sig")

def triage(content: str) -> dict:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "다음 고객 문의를 분석해 JSON으로만 답하라.\n"
                    "키: category(배송/환불/교환/결제/상품문의/칭찬/불만 중 하나), "
                    "urgent(true/false), summary(20자 이내), "
                    "suggested_reply(고객에게 보낼 추천 답변 1문장)"  # ← 추가한 키
                )
            },
            {
                "role": "user",
                "content": f"문의: {content}"
            }
        ],
        temperature=0,
        response_format={"type": "json_object"},  # ← JSON 강제
    )
    return json.loads(resp.choices[0].message.content)

# 앞 15건에 적용 → 긴급 건만 출력
print("=== 긴급 문의 (urgent=True) ===")
for content in df["content"].head(15):
    r = triage(content)
    if r.get("urgent"):
        print(f"- [{r['category']}] {content}")
        print(f"    추천답변: {r['suggested_reply']}")

=== 긴급 문의 (urgent=True) ===
- [결제] 카드 결제가 두 번 청구된 것 같아요. 확인 부탁드립니다.
    추천답변: 죄송합니다. 중복 결제 확인 후 금액을 환불해 드리겠습니다.
- [배송] 주문한 지 5일이 지났는데 아직도 배송중이에요. 언제 도착하나요?
    추천답변: 현재 배송 중이며, 추적 번호를 확인해 주시면 3~5일 이내 도착 예정입니다.
